# SMAC via OptunaHub

Notebook for the course [Master Hyperparameter Optimization for Tabular Learning](http://www.trainindata.com/p/master-hyperparameter-optimization-for-tabular-learning)

[OptunaHub](https://hub.optuna.org/) is a community registry of samplers, pruners, and visualizations built on top of Optuna's public APIs, but not shipped with Optuna itself. In this notebook we load its [SMAC3 sampler](https://hub.optuna.org/samplers/smac_sampler/), a wrapper around [SMAC3](https://automl.github.io/SMAC3/), a well-established Bayesian optimization library from AutoML.org.

By default, SMAC uses a **Random Forest** surrogate model (instead of the Parzen estimators TPE uses, or the Gaussian process GPSampler uses) together with expected improvement and an aggressive racing mechanism to decide which of two configurations performs better with fewer evaluations.

**A caveat worth knowing before you reach for it**: unlike `TPESampler` or `GPSampler`, the OptunaHub `SMACSampler` needs its **entire search space declared upfront**, as a fixed dictionary of `optuna.distributions`. It doesn't support the fully dynamic, define-by-run style used elsewhere in this section, where an objective function can call `trial.suggest_*` conditionally, inside `if` branches. Hence, this sampling algorithm is not useful for conditional hyperparameter spaces.

```bash
pip install optunahub smac
```


<div style="
    padding: 12px 16px;
    border-left: 5px solid #2196f3;
    background-color: #eaf4fd;
    border-radius: 4px;
">
<strong>Note:</strong>
At the time of creating this notebook, smac was not compatible with sklearn version 1.9.0. Hence, I am a bit hesitant at recommending this solution, since I don't really now how well supported and maintaned it is.
</div>

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier

import optuna
import optunahub

In [2]:
# load dataset

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
y = y.map({0: 1, 1: 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

X_train.shape, X_test.shape

((398, 30), (171, 30))

## Define the objective function

This is the same objective as notebook 01's RandomForest example.

In [3]:
def objective(trial):

    rf_n_estimators = trial.suggest_int("rf_n_estimators", 100, 1000)
    rf_criterion = trial.suggest_categorical("rf_criterion", ['gini', 'entropy'])
    rf_max_depth = trial.suggest_int("rf_max_depth", 1, 4)
    rf_min_samples_split = trial.suggest_float("rf_min_samples_split", 0.01, 1)

    model = RandomForestClassifier(
        n_estimators=rf_n_estimators,
        criterion=rf_criterion,
        max_depth=rf_max_depth,
        min_samples_split=rf_min_samples_split,
    )

    score = cross_val_score(model, X_train, y_train, cv=3)
    accuracy = score.mean()
    return accuracy

## Declare the static search space for SMAC

Same parameter names and ranges as the `trial.suggest_*` calls above -- SMAC uses this dictionary to build its internal configuration space, then Optuna's trial mechanism reads back the value SMAC picked for each parameter when the objective function runs.

In [4]:
search_space = {
    "rf_n_estimators": optuna.distributions.IntDistribution(100, 1000),
    "rf_criterion": optuna.distributions.CategoricalDistribution(["gini", "entropy"]),
    "rf_max_depth": optuna.distributions.IntDistribution(1, 4),
    "rf_min_samples_split": optuna.distributions.FloatDistribution(0.01, 1),
}

In [5]:
# load the SMAC sampler from OptunaHub's registry

module = optunahub.load_module("samplers/smac_sampler")
SMACSampler = module.SMACSampler

sampler = SMACSampler(
    search_space,
    n_trials=20,
    output_directory="smac3_output",
)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
)

study.optimize(objective, n_trials=20)

[INFO][abstract_initial_design.py:143] Using 5 initial design configurations and 0 additional configurations.


[I 2026-08-26 00:37:06,740] A new study created in memory with name: no-name-78da0177-66c8-4572-bb34-a644a58e0be9


[INFO][abstract_intensifier.py:313] Using only one seed for deterministic scenario.


[INFO][abstract_intensifier.py:523] Added config 590dcf as new incumbent because there are no incumbents yet.


[I 2026-08-26 00:37:07,404] Trial 0 finished with value: 0.6256360598465861 and parameters: {'rf_n_estimators': 629, 'rf_criterion': np.str_('gini'), 'rf_max_depth': 2, 'rf_min_samples_split': 0.6796816055756}. Best is trial 0 with value: 0.6256360598465861.


...

[I 2026-08-26 00:37:21,832] Trial 15 finished with value: 0.9271094402673349 and parameters: {'rf_n_estimators': 135, 'rf_criterion': np.str_('gini'), 'rf_max_depth': 3, 'rf_min_samples_split': 0.3070535484044}. Best is trial 14 with value: 0.9472544998860788.


[I 2026-08-26 00:37:23,064] Trial 16 finished with value: 0.9346851978430926 and parameters: {'rf_n_estimators': 200, 'rf_criterion': np.str_('gini'), 'rf_max_depth': 4, 'rf_min_samples_split': 0.1909469881234}. Best is trial 14 with value: 0.9472544998860788.


[I 2026-08-26 00:37:24,443] Trial 17 finished with value: 0.9396977291714134 and parameters: {'rf_n_estimators': 298, 'rf_criterion': np.str_('gini'), 'rf_max_depth': 4, 'rf_min_samples_split': 0.1980302264477}. Best is trial 14 with value: 0.9472544998860788.


[I 2026-08-26 00:37:24,883] Trial 18 finished with value: 0.6256360598465861 and parameters: {'rf_n_estimators': 144, 'rf_criterion': np.str_('entropy'), 'rf_max_depth': 4, 'rf_min_samples_split': 0.7755752633152}. Best is trial 14 with value: 0.9472544998860788.


[I 2026-08-26 00:37:25,689] Trial 19 finished with value: 0.9146350725298094 and parameters: {'rf_n_estimators': 429, 'rf_criterion': np.str_('entropy'), 'rf_max_depth': 1, 'rf_min_samples_split': 0.595960921204}. Best is trial 14 with value: 0.9472544998860788.


# Analyze results

In [6]:
study.best_params

{'rf_n_estimators': 207,
 'rf_criterion': 'gini',
 'rf_max_depth': 4,
 'rf_min_samples_split': 0.1917557058481}

In [7]:
study.best_value

0.9472544998860788

Compare `study.best_value` here against the Random Search, TPE, CMA-ES and GPSampler results in notebook 01 -- same model, same data, same number of trials, different samplers.